# 03 Data binding

Phase 3 turns a style into colour and height arrays plus a `Scale`. The map below is drawn with plain Lonboard; Phase 4 will do this inside kharita.

In [1]:
import warnings

import geopandas as gpd
import pandas as pd
from lonboard import Map, PolygonLayer

import kharita as kh

warnings.filterwarnings("ignore", module="lonboard")
states = gpd.read_file("../data/processed/sudan_states.gpkg")
b = kh.bind(states, "fill-[pop/area_km2] ramp-ylorrd scheme-q5")
b.fill.shape, b.fill.dtype

((18, 4), dtype('uint8'))

In [2]:
# The Scale the legend will be drawn from
s = b.fill_scale
pd.DataFrame({"class": s.labels, "colour": s.colors, "states": s.counts})

,class,colour,states
0,1.9 to 9.5,#ffffcc,4
1,9.5 to 16.7,#fee187,3
2,16.7 to 24.9,#fd8c3c,4
3,24.9 to 38.3,#d41020,3
4,38.3 to 248.9,#800026,4


In [3]:
# Same data, three schemes: see how the story changes
rows = []
for style in ["scheme-q5", "scheme-e5", "scheme-[10,25,50,100]"]:
    sc = kh.bind(states, f"fill-[density] ramp-ylorrd {style}").fill_scale
    rows.append({"scheme": sc.scheme, "states per class": sc.counts, "classes": sc.labels})
pd.DataFrame(rows)

,scheme,states per class,classes
0,"quantiles, 5 classes","(4, 3, 4, 3, 4)","(1.9 to 9.5, 9.5 to 16.7, 16.7 to 24.9, 24.9 t..."
1,"equal interval, 5 classes","(16, 0, 1, 0, 1)","(2 to 51, 51 to 101, 101 to 150, 150 to 199, 1..."
2,"breaks at 10, 25, 50, 100","(4, 7, 5, 0, 2)","(under 10, 10 to 25, 25 to 50, 50 to 100, 100 ..."


In [4]:
# Preview with plain Lonboard using kharita's colours
layer = PolygonLayer.from_geopandas(
    states[["geometry"]],
    get_fill_color=b.fill,
    get_line_color=[255, 255, 255, 255],
    get_line_width=1,
    line_width_units="pixels",
)
Map(layer)

In [5]:
# Heights: the tallest state is a tenth of the country's extent unless extrude-max is set
h = kh.bind(states, "fill-[density] ramp-ylorrd extrude-[density]")
print(f"tallest: {h.elevation_max / 1000:,.0f} km")
ext = PolygonLayer.from_geopandas(states[["geometry"]], get_fill_color=h.fill, extruded=True, get_elevation=h.elevation)
Map(ext, view_state={"longitude": 30, "latitude": 13, "zoom": 4.3, "pitch": 45, "bearing": -10})

tallest: 179 km


In [7]:
# Wireframe keeps state edges visible; viridis holds up better under 3D lighting than ylorrd's pale end
h = kh.bind(states, "fill-[density] ramp-viridis extrude-[density]")
ext = PolygonLayer.from_geopandas(
    states[["geometry"]],
    get_fill_color=h.fill,
    extruded=True,
    get_elevation=h.elevation,
    wireframe=True,
    get_line_color=[255, 255, 255, 120],
)
Map(ext, view_state={"longitude": 30, "latitude": 13, "zoom": 4.3, "pitch": 45, "bearing": -10})

In [6]:
# Warnings kharita raises about the data, not the syntax
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", kh.KharitaWarning)
    demo = states.assign(code=[1, 2, 3] * 6, area_zero=0.0)
    kh.bind(demo, "fill-[code]")
    try:
        kh.bind(demo, "fill-[pop/area_zero]")
    except kh.BindingError as err:
        print("BindingError:", err)
for w in caught:
    print("Warning:", w.message)

BindingError: [pop / area_zero] has no values to colour: every row is missing.
